## 1. Tiền xử lý

In [1]:
from pathlib import Path
import sys

project = Path.cwd()
if project.name == "notebooks":
    project = project.parent
sys.path.insert(0, str(project))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from src.evaluation import DATA, RESULTS, SEED, load_data, make_folds, save_submission
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from src.features import add_features, preprocess

X, y_log, X_test, test_ids, sample = load_data()
processor = preprocess(scale=False)


## 2. Tạo đặc trưng 

In [2]:
feature_sets = {
    "basic": (X, X_test),
    "engineered": (add_features(X), add_features(X_test)),
}
folds = list(make_folds().split(X))


## 3. Huấn luyện và tuning

In [4]:
%pip install catboost

  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached plotly-7.1.0-py3-none-any.whl.metadata (8.8 kB)
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   - -------------------------------------- 4.2/100.2 MB 19.4 MB/s eta 0:00:05
   --- ------------------------------------ 7.6/100.2 MB 21.3 MB/s eta 0:00:05
   ---- ----------------------------------- 11.8/100.2 MB 19.4 MB/s eta 0:00:05
   ------ --------------------------------- 15.5/100.2 MB 17.7 MB/s eta 0:00:05
   ------ --------------------------------- 16.8/100.2 MB 15.8 MB/s eta 0:00:06
   ------- -------------------------------- 17.6/100.2 MB 14.0 MB/s eta 0:00:06
   ------- -------------------------------- 18.4/100.2 MB 12.7 MB/s eta 0:00:07
   ------- -------------------------------- 19.4/100.2 MB 11.3 MB/s eta 0:00:08
   -------- ------------------------------- 20.2/100.2 MB 10.6 MB/s eta 0:00:08
   -------- --


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from catboost import CatBoostRegressor

name = "catboost"
pipeline = Pipeline([("preprocess", processor), ("model", CatBoostRegressor(iterations=400, learning_rate=0.05, random_seed=SEED, thread_count=2, verbose=False, allow_writing_files=False))])
grid = {"model__depth": [3, 4, 5]}
scoring = "neg_root_mean_squared_error"

# So sánh đặc trưng với cùng mô hình và tham số.
ablation = {
    label: -cross_val_score(pipeline, train_features, y_log, cv=folds, scoring=scoring)
    for label, (train_features, _) in feature_sets.items()
}
feature_choice = min(ablation, key=lambda label: ablation[label].mean())
train_features, test_features = feature_sets[feature_choice]
search = GridSearchCV(pipeline, grid, cv=folds, scoring=scoring, error_score="raise")
search.fit(train_features, y_log)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...bose=False))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__depth': [3, 4, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.","[(array([ 0, ...shape=(1168,)), ...), (array([ 0, ...shape=(1168,)), ...), ...]"
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to 


### Nhận xét về quá trình huấn luyện và tối ưu mô hình CatBoost

- Mô hình sử dụng: Nhóm áp dụng thuật toán `CatBoostRegressor` để dự đoán giá bán nhà. Đây là mô hình kết hợp nhiều cây quyết định, có khả năng học các mối quan hệ phức tạp giữa đặc trưng đầu vào và giá bán.
- Tiền xử lý dữ liệu: Sử dụng `ColumnTransformer` để xử lý riêng các biến số và biến phân loại. Các giá trị bị thiếu được bổ sung bằng `SimpleImputer`, trong khi các biến phân loại được mã hóa bằng `OneHotEncoder` để phù hợp với dữ liệu đầu vào của mô hình.
- Quy trình huấn luyện: Các bước tiền xử lý và mô hình `CatBoostRegressor` được kết hợp trong `Pipeline`, giúp thực hiện nhất quán quá trình xử lý dữ liệu và huấn luyện.
- Tối ưu siêu tham số: Sử dụng `GridSearchCV` để tìm kiếm cấu hình siêu tham số phù hợp trong các giá trị được thiết lập, nhằm lựa chọn mô hình có hiệu quả dự đoán tốt nhất theo tiêu chí đánh giá.
- Kết quả hiển thị: Hệ thống đã trả về cấu trúc mô hình tốt nhất dưới dạng `best_estimator_`, bao gồm các bước tiền xử lý và mô hình CatBoost. Tuy nhiên, sơ đồ chưa thể hiện giá trị siêu tham số tối ưu hoặc sai số dự đoán cụ thể.

**Kết luận:** Quá trình huấn luyện đã kết hợp tiền xử lý dữ liệu và tối ưu siêu tham số trong một quy trình thống nhất. Cấu hình tốt nhất được lựa chọn thông qua `GridSearchCV`, nhưng cần xem thêm kết quả đánh giá để xác định hiệu quả dự đoán thực tế của mô hình.


## 4. Đánh giá và xuất CSV

In [6]:
import json

rows = []
for variant, scores in ablation.items():
    rows.append({"model": name, "variant": variant, "features": variant, "params": "default", **{
        f"fold_{i}": score for i, score in enumerate(scores, 1)
    }})
tuned_scores = [-search.cv_results_[f"split{i}_test_score"][search.best_index_] for i in range(len(folds))]
rows.append({"model": name, "variant": "tuned", "features": feature_choice,
             "params": json.dumps(search.best_params_), **{
                 f"fold_{i}": score for i, score in enumerate(tuned_scores, 1)
             }})
scores = pd.DataFrame(rows)
fold_columns = [f"fold_{i}" for i in range(1, len(folds) + 1)]
scores["cv_mean"] = scores[fold_columns].mean(axis=1)
scores["cv_std"] = scores[fold_columns].std(axis=1, ddof=0)
scores["selected"] = False
best = scores["cv_mean"].idxmin()
scores.loc[best, "selected"] = True
selected = scores.loc[best]
if selected["variant"] == "tuned":
    final_model = search.best_estimator_
else:
    train_features, test_features = feature_sets[selected["features"]]
    final_model = clone(pipeline).fit(train_features, y_log)
submission = save_submission(name, final_model.predict(test_features), test_ids, sample)
cv_dir = RESULTS / "cv"
cv_dir.mkdir(parents=True, exist_ok=True)
scores.to_csv(cv_dir / f"{name}.csv", index=False)
display(scores)
display(submission.head())


,model,variant,features,params,fold_1,fold_2,fold_3,fold_4,fold_5,cv_mean,cv_std,selected
0,catboost,basic,basic,default,0.128449,0.109556,0.152203,0.127535,0.106117,0.124772,0.016451,False
1,catboost,engineered,engineered,default,0.128782,0.107209,0.150902,0.127122,0.106579,0.124119,0.016381,False
2,catboost,tuned,engineered,"{""model__depth"": 5}",0.127020,0.108276,0.154381,0.123199,0.106531,0.123881,0.017234,True


,Id,SalePrice
0,1461,125328.358975
1,1462,158343.653649
2,1463,188640.900448
3,1464,194973.924953
4,1465,185123.306635


### Nhận xét. 
Trước khi tinh chỉnh, bộ đặc trưng bổ sung giúp RMSE trung bình qua 5 fold giảm nhẹ từ 0,124772 xuống 0,124119. Sau khi tinh chỉnh trên bộ bổ sung, cấu hình depth = 5 đạt RMSE trung bình thấp nhất trong ba cấu hình đã đánh giá, bằng 0,123881. Độ lệch chuẩn giữa các fold tăng nhẹ từ 0,016381 lên 0,017234. Nhóm chọn cấu hình này để dự đoán giá bán trên tập test và xuất tệp nộp Kaggle.